# # manmit get data response

In [ ]:
import pandas as pd
import json

# Data JSON Anda (dalam bentuk list of dictionary)
with open("dist/temp.json", "r", encoding="utf-8") as file:
    data_json = json.load(file)

# 1. Gunakan json_normalize untuk otomatis membongkar nested dictionary (mitra_detail)
df_full = pd.json_normalize(data_json)

# 2. Pilih kolom utama yang diinginkan dan gabungkan dengan semua kolom hasil bongkaran mitra_detail
kolom_utama = ["id_ms", "kd_survei", "kd_kab", "id_kegiatan"]
kolom_mitra_detail = [col for col in df_full.columns if col.startswith("mitra_detail.")]

df_final = df_full[kolom_utama + kolom_mitra_detail]

# 3. Opsional: Bersihkan nama kolom agar teks "mitra_detail." hilang
df_final.columns = [col.replace("mitra_detail.", "") for col in df_final.columns]

# Tampilkan DataFrame
df_final


In [3]:
df_final.to_csv('temp.csv')

In [6]:
len("PUTU AYU AZARIA WULAN THAJANI")

29

# # cocard manmit

## mindah ke folder berdasarkan mitra

In [ ]:
# masukin ke folder terpilih dari folder cocard di download dengan user ASUS dan
import os
import shutil
import pandas as pd

# 1. Baca data CSV kamu
df = pd.read_csv("sobatid.csv")
df['hasil']=''
df['log']=''

# 2. Ambil daftar nama file dari kolom CSV (sesuaikan nama kolomnya)
# Pastikan data berupa string dan hilangkan spasi yang tidak sengaja terikut
nama_cari = df["sobatid"].astype(str).str.strip().tolist()

# 3. Tentukan jalur (path) folder kamu
folder_sumber = r"D:\2026\SE\cocard\terpilih"
folder_tujuan = r"D:\2026\SE\cocard\terpilihv2"

# Buat folder tujuan otomatis jika belum ada
os.makedirs(folder_tujuan, exist_ok=True)

# Ambil semua daftar file asli yang ada di folder sumber sekali saja
isi_folder_sumber = os.listdir(folder_sumber)

# 2. Loop berdasarkan data di DataFrame CSV
for indeks, baris in df.iterrows():
    # Ambil kata kunci dari kolom CSV (sesuaikan nama 'nama_kolom_csv')
    kata_cari = str(baris["sobatid"]).strip()

    # Lewati jika baris kosong atau NaN
    if not kata_cari or kata_cari == "nan":
        continue

    file_ditemukan = False

    # 3. Cek ke semua file di folder sumber
    for nama_file in isi_folder_sumber:
        # Cek apakah file berupa PNG dan mengandung kata dari CSV
        if nama_file.lower().endswith(".png") and (kata_cari in nama_file):
            jalur_asal = os.path.join(folder_sumber, nama_file)
            jalur_baru = os.path.join(folder_tujuan, nama_file)

            # Pindahkan file
            shutil.move(jalur_asal, jalur_baru)
            log = f"[{kata_cari}] -> Berhasil memindahkan: {nama_file}"
            print(log)
            df.at[indeks, 'hasil'] = 'found'
            df.at[indeks, 'log'] = log


            # Perbarui daftar agar file yang sudah pindah tidak dicek lagi
            isi_folder_sumber.remove(nama_file)
            file_ditemukan = True
            break  # Keluar dari loop file jika sudah ketemu dan pindah

    # 4. Print jika tidak ditemukan
    if not file_ditemukan:
        log = f"[{kata_cari}] -> Tidak ditemukan"
        print(log)
        df.at[indeks, 'hasil'] = 'notfound'
        df.at[indeks, 'log'] = log

df.to_csv('sobatid.csv')

[510326050099] -> Tidak ditemukan
[510326050050] -> Tidak ditemukan
[510326050080] -> Tidak ditemukan
[510326050052] -> Tidak ditemukan
[510326050097] -> Tidak ditemukan
[510326050070] -> Tidak ditemukan
[510326050092] -> Tidak ditemukan
[510326050330] -> Tidak ditemukan
[510326050094] -> Tidak ditemukan
[510326050071] -> Tidak ditemukan
[510326050090] -> Tidak ditemukan
[510326050091] -> Tidak ditemukan
[510326050077] -> Tidak ditemukan
[510326050064] -> Tidak ditemukan
[510326050075] -> Tidak ditemukan
[510326050263] -> Tidak ditemukan


## generate url cek mitra

In [ ]:
id_ms = "469568"
id_mitra = "426918"
kd_survei = "SE2026"
id_kegiatan = "66"
kd_prov = "51"

In [ ]:

# def generate_url_code(t, kd_survei, id_keg, kd_prov):
def generate_url_code(instance,var=[]):
    '''var=["id_ms", "id_mitra", "kd_survei", 'id_keg', 'kd_prov']'''

    import json
    from lzstring import LZString
    allowed_keys = ["id_ms", "id_mitra", "kd_survei", 'id_keg', 'kd_prov']
    a = dict(zip(allowed_keys, var))

    # Membuat format string yang di-concat seperti di JS
    concat_str = f"{a.get('id_ms')},{a.get('id_mitra')},{a['kd_survei']},{a['id_keg']},{a['kd_prov']}"

    # Ubah string tersebut menjadi format JSON string
    # JS: JSON.stringify(...)
    json_str = json.dumps(concat_str)

    # Kompresi menggunakan LZString ke format Encoded URI Component
    lz = LZString()
    compressed = lz.compressToEncodedURIComponent(json_str)

    return f"https://mitra.bps.go.id/c/{compressed}"


In [ ]:
url_hasil = generate_url_code([id_ms, id_mitra, kd_survei, id_kegiatan, kd_prov])
url_hasil

'https://mitra.bps.go.id/c/EQFgbAnArGAcA0IBMkCMCDKBRJAGF8YY8UqwQA'

## export pdf to png and rename it

In [ ]:
import os
import csv
path_poppler_windows = r"C:\poppler-26.02.0\Library\bin" # Sesuaikan dengan versi yang Anda unduh
from pdf2image import convert_from_path

# 1. Konfigurasi File & Folder
pdf_path = "pelatihanSE/cocard-finish.pdf"
csv_path = "pelatihanSE/kepka.csv"
output_dir = "pelatihanSE/cocard-finish"

os.makedirs(output_dir, exist_ok=True)

# 2. Baca data dari CSV dan simpan ke dalam List
daftar_nama_baru = []
with open(csv_path, mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    for row in reader:
        # Menggabungkan kolom id dan nama menjadi nama file baru
        # Contoh: "101_Invoice_Januari.png"
        nama_file = f"{row['nama']}_{row['sobat_id']}.png"
        daftar_nama_baru.append(nama_file)

# 3. Export PDF dan langsung namai berdasarkan CSV
print(f"Membuka PDF: {pdf_path}...")
pages = convert_from_path(pdf_path, poppler_path=path_poppler_windows)

print("Mengekstrak halaman dan mengganti nama sesuai CSV...")
for i, page in enumerate(pages):
    # Jika baris CSV lebih sedikit dari jumlah halaman PDF, gunakan nama default
    if i < len(daftar_nama_baru):
        nama_akhir = daftar_nama_baru[i]
    else:
        nama_akhir = f"halaman_{i + 1}_tanpa_nama_csv.png"
        print(f"Peringatan: Baris CSV habis. Halaman {i+1} menggunakan nama default.")
    
    # Simpan file langsung dengan nama baru
    path_simpan = os.path.join(output_dir, nama_akhir)
    page.save(path_simpan, 'PNG')
    print(f"Tersimpan: {nama_akhir}")

print(f"\nSelesai! {len(pages)} halaman berhasil diproses di folder '{output_dir}'.")

## or generate image data from template.png (mailmerge)

In [2]:
from PIL import Image, ImageDraw, ImageFont
import qrcode
import textwrap
import os

def generate_custom_document(nama, data_qr, config, output_name):
    cfg = config
    
    # --- 1. PROSES LOAD IMAGE & FONT ---
    try:
        # 🌟 UBAH: Gunakan mode RGBA untuk menjaga channel transparansi agar warna tidak rusak/kuning
        img = Image.open(cfg["bg_path"]).convert("RGBA")
        draw = ImageDraw.Draw(img)
        
        # Coba load font utama (Raleway)
        font = ImageFont.truetype(cfg['font_type'], cfg["font_size"])
        print("ℹ️ Mencoba memuat font Raleway...")
    except Exception as e:
        # 🌟 UBAH: Jika file font utama corrupt/gagal, otomatis fallback ke Arial bawaan Windows
        print(f"⚠️ Font Raleway bermasalah, otomatis dialihkan ke Arial Windows. Detail: {e}")
        try:
            font = ImageFont.truetype(r"C:\Windows\Fonts\arial.ttf", cfg["font_size"])
        except:
            # Jika Arial tidak ketemu (misal bukan di Windows), gunakan font default bawaan Pillow
            font = ImageFont.load_default()

    # --- 2. LOGIKA NAMA & GAMBAR TEKS ---
    words = nama.split()
    if len(nama) > 35: 
        nama = f"{' '.join(words[:-1])} {words[-1]}."
    
    lines = textwrap.wrap(nama, width=15)
    
    x1, y1, w_box, h_box = cfg["text_box"]
    x_center = x1 + (w_box / 2) 
    y_current = y1

    for line in lines:
        bbox = draw.textbbox((0, 0), line, font=font)
        line_height = bbox[3] - bbox[1]
        
        draw.text((x_center, y_current), line, fill="black", font=font, anchor="ma")
        y_current += line_height + 10 

    # --- 3. GAMBAR QR CODE ---
    qr = qrcode.make(data_qr)
    
    # 🌟 UBAH: Konversi QR ke mode RGBA agar sinkron dengan mode gambar background
    qr = qr.convert("RGBA") 
    
    qr_width = cfg["qr_box"][2]
    qr_height = cfg["qr_box"][3]
    qr = qr.resize((qr_width, qr_height))
    
    qx = cfg["qr_box"][0] + (qr_width - qr.size[0]) // 2
    qy = cfg["qr_box"][1] + (qr_height - qr.size[1]) // 2
    
    # 🌟 UBAH: Tambahkan parameter mask=qr agar warna hitam-putih QR terisolasi dan tidak berubah kuning
    img.paste(qr, (qx, qy), mask=qr)
    
    # Konversi balik ke RGB sebelum disimpan sebagai gambar final
    final_img = img.convert("RGB")
    final_img.save(output_name)
    print(f"✅ Sukses! Dokumen disimpan sebagai: {output_name}")

    # loerm

In [3]:
# path
# BASE_DIR = os.path.dirname(os.path.abspath(__file__))

# Contoh config untuk template yang berbeda
templates = {
    "id_card": {
        # "bg_path": os.path.join(BASE_DIR, "pelatihanSE", "cocard-template.png"),
        "bg_path": r"D:\jim\auto-fasih-sm\pelatihanSE\cocard-template.png",
        "text_box": (138, 813, 200, 970), # start x, start y, width box, len box
        "qr_box": (460, 1148, 330, 330),
        "font_size": 27,
        # "font_type": os.path.join(BASE_DIR, "pelatihanSE", "raleway-medium.ttf")
        "font_type": r"D:\jim\auto-fasih-sm\pelatihanSE\raleway-medium.ttf"
        # "font_type": "arial.ttf"
    }
}

# Cara Pakai:
generate_custom_document("Jey Neutron", "link_data", templates["id_card"], "hasil_cocard.png")


ℹ️ Mencoba memuat font Raleway...
✅ Sukses! Dokumen disimpan sebagai: hasil_cocard.png


In [ ]:
from PIL import Image, ImageDraw, ImageFont
import qrcode
import textwrap

def gen_mergemail(nama, data_qr, config, output_name):
    cfg = config
    
    try:
        # Load background RGBA agar warna QR terkunci hitam-putih
        img = Image.open(cfg["bg_path"]).convert("RGBA")
        draw = ImageDraw.Draw(img)
        font = ImageFont.truetype(cfg['font_type'], cfg["font_size"])
    except Exception as e:
        print(f"❌ GAGAL LOAD FILE! Detail: {e}")
        return 

    # 1. Bersihkan String Nama
    nama_clean = str(nama).strip().upper()
    
    x1, y1, w_box, h_box = cfg["text_box"]
    x_center = img.width / 2  
    y_floor = y1  
    
    # Gunakan textwrap dengan width lebih besar agar kata tidak terpotong per huruf
    if len(nama_clean) > 35:
        nama_potong = nama_clean[:35]
        words = nama_potong.split()
        if len(words) > 1:
            all_but_last = " ".join(words[:-1])
            nama_final = f"{all_but_last} {words[-1][0]}."
        else:
            nama_final = nama_clean[:20] # Fallback jika hanya 1 kata panjang
        # Jika lebih dari 35 karakter, paksa potong lebih pendek agar otomatis jadi 2 baris yang seimbang
        lines = textwrap.wrap(nama_final, width=20)
    else:
        # Jika di bawah 35 karakter, berikan ruang lebar agar tetap aman dalam 1 baris
        lines = textwrap.wrap(nama_clean, width=30)
    text_to_draw = "\n".join(lines)
    
    y_start = y1 - (cfg["font_size"] * (len(lines) - 1) + 20)
    
    # UBAH: Menggunakan anchor="ma" (Middle-Top) dikombinasikan dengan hitungan y_start dinamis di atas
    draw.text((x_center, y_start), text_to_draw, fill="black", font=font, anchor="ma", align="center")

    # 3. Gambar QR Code (Hitam-Putih Bersih)
    qr = qrcode.make(data_qr).convert("RGBA") 
    qr_width = cfg["qr_box"][2]
    qr_height = cfg["qr_box"][3]
    qr = qr.resize((qr_width, qr_height))
    
    qx = cfg["qr_box"][0] + (qr_width - qr.size[0]) #// 2
    qy = cfg["qr_box"][1] + (qr_height - qr.size[1]) #// 2
    img.paste(qr, (qx, qy), mask=qr)
    
    # Simpan instan dalam hitungan milidetik
    final_img = img.convert("RGB")
    final_img.save(output_name)
    print(f"⚡ Sukses Generate: {output_name}")

In [ ]:
# --- KONFIGURASI BARU ---
templates = {
    "id_card": {
        # "bg_path": r"D:\jim\auto-fasih-sm\pelatihanSE\cocard-template.png",
        "bg_path": r"pelatihanSE\cocard-template.png",
        "text_box": (200, 955, 600, 970), # start x, start y, width box, height box; sementara yg kepake baru y aja
        "qr_box": (450, 1148, 330, 330),
        "font_size": 80, 
        # "font_type": r"C:\Windows\Fonts\arial.ttf" 
        # "font_type": r"D:\jim\auto-fasih-sm\raleway-medium.ttf"
        "font_type": r"pelatihanSE\Raleway-SemiBold.ttf"
    }
}

# try Jalankan fungsi super cepat 
gen_mergemail("Lorem ipsum dolor", "link_data", templates["id_card"], "temphasil.png")
gen_mergemail("Lorem ipsum dolor sit amet lorem loreman", "https://mitra.bps.go.id/c/EQdhBYCZwNgGgMwE4AcyUjgZQKKQAyTwzwCsAjMEA", templates["id_card"], "temphasil2.png")



⚡ Sukses Generate: temphasil.png
⚡ Sukses Generate: temphasil2.png


# # fasih assign


## get error json as csv

In [9]:
import pandas as pd
import json

# 1. Struktur data JSON Anda (sudah dilengkapi penutupnya)
with open("errors.json", "r", encoding="utf-8") as file:
    raw_json = json.load(file)

# 2. Ambil data langsung dari dalam key 'content'
target_data = raw_json["data"]["content"]

# 3. Ubah menjadi DataFrame
df = pd.DataFrame(target_data)

# 4. Opsional: Bersihkan kolom 'users' agar tidak membawa tanda kurung siku [] di CSV
df["users"] = df["users"].apply(lambda x: ", ".join(x) if isinstance(x, list) else x)

# 5. Simpan ke file CSV
df.to_csv("error.json.csv", index=False)
print("Data JSON berhasil dikonversi ke error.json.csv!")

Data JSON berhasil dikonversi ke error.json.csv!


## get mitra assigned

In [ ]:
import json
import pandas as pd

# 1. Membaca file JSON
# (Asumsi teks JSON Anda dibungkus tanda kurung kurawal agar valid)
with open("by-user.json", "r") as file:
    payload = json.load(file)

# 2. Mengambil data utama dari dalam key 'data' -> 'content'
content_data = payload["data"]["content"]

# 3. Mengubah ke DataFrame dan membongkar list 'regions' di dalamnya
df = pd.json_normalize(
    content_data,
    record_path=["regions"],
    meta=["userId", "roleId", "totalRegions", "username"],
)

# 4. Memilih dan mengurutkan kolom sesuai kebutuhan Anda
df_hasil = df[["userId", "roleId", "totalRegions", "regionCode", "username"]]

# Tampilkan hasil
df_hasil
